In [7]:
import os

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import (
    KernelFunctionSelectionStrategy,
    KernelFunctionTerminationStrategy,
)
from semantic_kernel.kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import AuthorRole, ChatMessageContent, ChatHistoryTruncationReducer
from semantic_kernel.functions import KernelFunctionFromPrompt

In [8]:
def _create_kernel_with_chat_completion() -> Kernel:
    kernel = Kernel()
    client = AsyncOpenAI(
        api_key=os.environ.get("GITHUB_TOKEN"),
        base_url="https://models.inference.ai.azure.com/",
    )
    kernel.add_service(
        OpenAIChatCompletion(
            ai_model_id="gpt-4.1",
            async_client=client,
        )
    )
    return kernel

In [ ]:
async def main():
    # Shared kernel
    kernel = _create_kernel_with_chat_completion()

    REVIEWER_NAME = "Concierge"
    FRONTDESK_NAME = "FrontDesk"

    REVIEWER_INSTRUCTIONS = """
You are a hotel concierge focused on authentic local experiences.
Approve the Front Desk recommendation only when it is truly local and non‑touristy.
If not approved, give brief, actionable refinement guidance without giving a concrete example itinerary.
"""
    agent_reviewer = ChatCompletionAgent(
        kernel=kernel,
        name=REVIEWER_NAME,
        instructions=REVIEWER_INSTRUCTIONS,
    )

    FRONTDESK_INSTRUCTIONS = """
You are a Front Desk Travel Agent known for brevity.
Provide exactly one concise activity recommendation per turn for the traveler.
Incorporate prior concierge refinement guidance.
No chit chat.
"""
    agent_writer = ChatCompletionAgent(
        kernel=kernel,
        name=FRONTDESK_NAME,
        instructions=FRONTDESK_INSTRUCTIONS,
    )

    # Termination uses only last message (token saving)
    termination_function = KernelFunctionFromPrompt(
        function_name="termination",
        prompt=f"""
Examine the last message. If the {REVIEWER_NAME} clearly approved a recommendation
(using words like "approved", "this is approved", "I approve"), respond with: yes
Otherwise respond with: no

LAST:
{{{{$lastmessage}}}}
""",
    )

    # Selection based only on last message
    selection_function = KernelFunctionFromPrompt(
        function_name="selection",
        prompt=f"""
Given LAST, output only the name of the next participant:

Participants:
- {REVIEWER_NAME}
- {FRONTDESK_NAME}

Rules:
- After user input -> {FRONTDESK_NAME}
- After {FRONTDESK_NAME} -> {REVIEWER_NAME}
- After {REVIEWER_NAME} -> {FRONTDESK_NAME}

LAST:
{{{{$lastmessage}}}}
""",
    )

    # Keep only last exchange for strategies
    history_reducer = ChatHistoryTruncationReducer(target_count=1)

    chat = AgentGroupChat(
        agents=[agent_reviewer, agent_writer],
        termination_strategy=KernelFunctionTerminationStrategy(
            agents=[agent_reviewer],
            function=termination_function,
            kernel=kernel,
            result_parser=lambda r: (str(r.value[0]).strip().lower() == "yes") if r.value and r.value[0] else False,
            history_variable_name="lastmessage",
            maximum_iterations=12,
            history_reducer=history_reducer,
        ),
        selection_strategy=KernelFunctionSelectionStrategy(
            initial_agent=agent_writer,  # FrontDesk speaks after user
            function=selection_function,
            kernel=kernel,
            result_parser=lambda r: (str(r.value[0]).strip()
                                     if r.value and r.value[0] else FRONTDESK_NAME),
            history_variable_name="lastmessage",
            history_reducer=history_reducer,
        ),
    )

    user_input = "I would like to go to Rio de Janeiro."
    await chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))
    print(f"# User: '{user_input}'")

    async for content in chat.invoke():
        print(f"# Agent - {content.name or '*'}: '{content.content}'")

    print(f"# IS COMPLETE: {chat.is_complete}")

await main()

ValidationError: 1 validation error for KernelFunctionFromPrompt
prompt_execution_settings
  Input should be a valid dictionary [type=dict_type, input_value={'allow_dangerously_set_content', True}, input_type=set]
    For further information visit https://errors.pydantic.dev/2.11/v/dict_type

In [ ]:
import asyncio
import os

from semantic_kernel import Kernel
from semantic_kernel.agents import AgentGroupChat, ChatCompletionAgent
from semantic_kernel.agents.strategies import (
    KernelFunctionSelectionStrategy,
    KernelFunctionTerminationStrategy,
)
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistoryTruncationReducer
from semantic_kernel.functions import KernelFunctionFromPrompt

"""
The following sample demonstrates how to create a simple,
agent group chat that utilizes a Reviewer Chat Completion
Agent along with a Writer Chat Completion Agent to
complete a user's task.
"""

# Define agent names
REVIEWER_NAME = "Reviewer"
WRITER_NAME = "Writer"


def create_kernel() -> Kernel:
    """Creates a Kernel instance with an Azure OpenAI ChatCompletion service."""
    client = AsyncOpenAI(
        api_key=os.environ.get("GITHUB_TOKEN"),
        base_url="https://models.inference.ai.azure.com/",
    )
    
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(
            ai_model_id="gpt-4.1",
            async_client=client,
        ))
    return kernel


async def main():
    # Create a single kernel instance for all agents.
    kernel = create_kernel()

    # Create ChatCompletionAgents using the same kernel.
    agent_reviewer = ChatCompletionAgent(
        kernel=kernel,
        name=REVIEWER_NAME,
        instructions="""
Your responsibility is to review and identify how to improve user provided content.
If the user has provided input or direction for content already provided, specify how to address this input.
Never directly perform the correction or provide an example.
Once the content has been updated in a subsequent response, review it again until it is satisfactory.

RULES:
- Only identify suggestions that are specific and actionable.
- Verify previous suggestions have been addressed.
- Never repeat previous suggestions.
""",
    )

    agent_writer = ChatCompletionAgent(
        kernel=kernel,
        name=WRITER_NAME,
        instructions="""
Your sole responsibility is to rewrite content according to review suggestions.
- Always apply all review directions.
- Always revise the content in its entirety without explanation.
- Never address the user.
""",
    )

    # Define a selection function to determine which agent should take the next turn.
    selection_function = KernelFunctionFromPrompt(
        function_name="selection",
        prompt=f"""
Examine the provided RESPONSE and choose the next participant.
State only the name of the chosen participant without explanation.
Never choose the participant named in the RESPONSE.

Choose only from these participants:
- {REVIEWER_NAME}
- {WRITER_NAME}

Rules:
- If RESPONSE is user input, it is {REVIEWER_NAME}'s turn.
- If RESPONSE is by {REVIEWER_NAME}, it is {WRITER_NAME}'s turn.
- If RESPONSE is by {WRITER_NAME}, it is {REVIEWER_NAME}'s turn.

RESPONSE:
{{{{$lastmessage}}}}
""",
    )

    # Define a termination function where the reviewer signals completion with "yes".
    termination_keyword = "yes"

    termination_function = KernelFunctionFromPrompt(
        function_name="termination",
        prompt=f"""
Examine the RESPONSE and determine whether the content has been deemed satisfactory.
If the content is satisfactory, respond with a single word without explanation: {termination_keyword}.
If specific suggestions are being provided, it is not satisfactory.
If no correction is suggested, it is satisfactory.

RESPONSE:
{{{{$lastmessage}}}}
""",
    )

    history_reducer = ChatHistoryTruncationReducer(target_count=5)

    # Create the AgentGroupChat with selection and termination strategies.
    chat = AgentGroupChat(
        agents=[agent_reviewer, agent_writer],
        selection_strategy=KernelFunctionSelectionStrategy(
            initial_agent=agent_reviewer,
            function=selection_function,
            kernel=kernel,
            result_parser=lambda result: str(result.value[0]).strip() if result.value[0] is not None else WRITER_NAME,
            history_variable_name="lastmessage",
            history_reducer=history_reducer,
        ),
        termination_strategy=KernelFunctionTerminationStrategy(
            agents=[agent_reviewer],
            function=termination_function,
            kernel=kernel,
            result_parser=lambda result: termination_keyword in str(result.value[0]).lower(),
            history_variable_name="lastmessage",
            maximum_iterations=10,
            history_reducer=history_reducer,
        ),
    )

    print(
        "Ready! Type your input, or 'exit' to quit, 'reset' to restart the conversation. "
        "You may pass in a file path using @<path_to_file>."
    )

    is_complete = False
    while not is_complete:
        print()
        user_input = input("User > ").strip()
        if not user_input:
            continue

        if user_input.lower() == "exit":
            is_complete = True
            break

        if user_input.lower() == "reset":
            await chat.reset()
            print("[Conversation has been reset]")
            continue

        # Try to grab files from the script's current directory
        if user_input.startswith("@") and len(user_input) > 1:
            file_name = user_input[1:]
            script_dir = os.path.dirname(os.path.abspath(__file__))
            file_path = os.path.join(script_dir, file_name)
            try:
                if not os.path.exists(file_path):
                    print(f"Unable to access file: {file_path}")
                    continue
                with open(file_path, "r", encoding="utf-8") as file:
                    user_input = file.read()
            except Exception:
                print(f"Unable to access file: {file_path}")
                continue

        # Add the current user_input to the chat
        await chat.add_chat_message(message=user_input)

        try:
            async for response in chat.invoke():
                if response is None or not response.name:
                    continue
                print()
                print(f"# {response.name.upper()}:\n{response.content}")
        except Exception as e:
            print(f"Error during chat invocation: {e}")

        # Reset the chat's complete flag for the new conversation round.
        chat.is_complete = False


if __name__ == "__main__":
    await main()

Ready! Type your input, or 'exit' to quit, 'reset' to restart the conversation. You may pass in a file path using @<path_to_file>.

